In [1]:
!pip install dagshub mlflow xgboost scikit-learn -q

In [2]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import dagshub
from mlflow.tracking import MlflowClient
warnings.filterwarnings('ignore')

dagshub.init(repo_owner='sophyrise', repo_name='ieee-cis-fraud-detection', mlflow=True)
client = MlflowClient()
print("✅ MLflow tracking URI:", mlflow.get_tracking_uri())

Accessing as sophyrise

Initialized MLflow to track repo "sophyrise/ieee-cis-fraud-detection"

Repository sophyrise/ieee-cis-fraud-detection initialized!

✅ MLflow tracking URI: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow


In [3]:
DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"

test_txn = pd.read_csv(f"{DATA_DIR}/test_transaction.csv")
test_id  = pd.read_csv(f"{DATA_DIR}/test_identity.csv")
test_id.columns = [c.replace('-', '_') for c in test_id.columns]

test = test_txn.merge(test_id, on="TransactionID", how="left")
transaction_ids = test["TransactionID"].copy()
X_test_raw = test.drop(columns=["TransactionID"])

del test, test_txn, test_id
gc.collect()

print(f"Test shape: {X_test_raw.shape}")
print(f"Transaction IDs: {len(transaction_ids)}")

Test shape: (506691, 432)
Transaction IDs: 506691


In [4]:
experiments = {
    "XGBoost":                     ("XGB_Final_Pipeline",  "XGBoost_FraudDetection"),
    "Random Forest":               ("RF_Final_Pipeline",   "RandomForest_FraudDetection"),
    "Decision Trees":              ("DT_Final_Pipeline",   "DecisionTree_FraudDetection"),
    "LogisticRegression-Training": ("LR_Final_Pipeline",   "LogisticRegression_FraudDetection"),
    "Neural Networks":             ("MLP_Final_Pipeline",  "MLP_FraudDetection"),
    "LightGBM":                    ("LGB_Final_Pipeline",  "LightGBM_FraudDetection"),
    "Gradient Boosting":           ("GB_Final_Pipeline",   "GradientBoosting_FraudDetection"),
}

results = []
for exp_name, (run_name, registry_name) in experiments.items():
    exp = client.get_experiment_by_name(exp_name)
    if exp is None:
        print(f"  ⚠️  not found: {exp_name}")
        continue

    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'",
        order_by=["start_time DESC"],
        max_results=1
    )
    if not runs:
        print(f"  ⚠️  run not found: {run_name} in {exp_name}")
        continue

    run  = runs[0]
    auc  = run.data.metrics.get("val_auc") or run.data.metrics.get("cv_mean_auc")
    metric_used = "val_auc" if "val_auc" in run.data.metrics else "cv_mean_auc"

    results.append({
        "model":         exp_name,
        "run_id":        run.info.run_id,
        "auc":           auc,
        "metric_used":   metric_used,
        "registry_name": registry_name,
    })
    print(f"  {exp_name:<35} | AUC: {auc:.4f} ({metric_used})")

results_df = pd.DataFrame(results).sort_values("auc", ascending=False).reset_index(drop=True)
print("\n📊 Model Ranking:")
print(results_df[["model", "auc", "metric_used"]].to_string())

best = results_df.iloc[0]
print(f"\n✅ Best model: {best['model']}")
print(f"   AUC: {best['auc']:.4f} | Run ID: {best['run_id']}")

  XGBoost                             | AUC: 0.9791 (val_auc)
  Random Forest                       | AUC: 0.9475 (val_auc)
  Decision Trees                      | AUC: 0.8667 (val_auc)
  LogisticRegression-Training         | AUC: 0.8807 (val_auc)
  Neural Networks                     | AUC: 0.8787 (val_auc)
  LightGBM                            | AUC: 0.9639 (val_auc)
  Gradient Boosting                   | AUC: 0.9200 (val_auc)

📊 Model Ranking:
                         model      auc metric_used
0                      XGBoost  0.97906     val_auc
1                     LightGBM  0.96392     val_auc
2                Random Forest  0.94747     val_auc
3            Gradient Boosting  0.91999     val_auc
4  LogisticRegression-Training  0.88069     val_auc
5              Neural Networks  0.87874     val_auc
6               Decision Trees  0.86667     val_auc

✅ Best model: XGBoost
   AUC: 0.9791 | Run ID: 97967f48e2fc4fd29de0f1865f833554


In [5]:
model_uri = f"models:/{best['registry_name']}/latest"
model = mlflow.pyfunc.load_model(model_uri)

print(f"✅ Model loaded successfully!")
print(f"   Loaded from registry: {best['registry_name']}")

✅ Model loaded successfully!
   Loaded from registry: XGBoost_FraudDetection


In [6]:
TXN_MISSING_THRESHOLD   = 0.80
ID_MISSING_THRESHOLD    = 0.95
NEAR_CONSTANT_THRESHOLD = 0.99
CORR_THRESHOLD          = 0.98

def preprocess(X_train, X_test, y_train):
    X_train = X_train.copy()
    X_test  = X_test.copy()

    id_cols  = [c for c in X_train.columns if c.startswith("id_") or c in ["DeviceType", "DeviceInfo"]]
    txn_cols = [c for c in X_train.columns if c not in id_cols]

    missing = X_train.isnull().mean()
    drop_missing = (
        [c for c in txn_cols if missing[c] > TXN_MISSING_THRESHOLD] +
        [c for c in id_cols  if missing[c] > ID_MISSING_THRESHOLD]
    )
    X_train = X_train.drop(columns=drop_missing)
    X_test  = X_test.drop(columns=[c for c in drop_missing if c in X_test.columns])

    near_const = [
        c for c in X_train.columns
        if X_train[c].value_counts(dropna=False, normalize=True).iloc[0] > NEAR_CONSTANT_THRESHOLD
    ]
    X_train = X_train.drop(columns=near_const)
    X_test  = X_test.drop(columns=[c for c in near_const if c in X_test.columns])

    for df in [X_train, X_test]:
        df["TransactionAmt_log"] = np.log1p(df["TransactionAmt"].clip(lower=0))
        df["hour_sin"] = np.sin(2 * np.pi * ((df["TransactionDT"] // 3600) % 24) / 24)
        df["hour_cos"] = np.cos(2 * np.pi * ((df["TransactionDT"] // 3600) % 24) / 24)
        df["day_of_week"] = (df["TransactionDT"] // 86400) % 7
    X_train = X_train.drop(columns=["TransactionDT"])
    X_test  = X_test.drop(columns=["TransactionDT"], errors="ignore")

    cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
    for c in cat_cols:
        mapping = {v: i for i, v in enumerate(X_train[c].astype(str).unique())}
        X_train[c] = X_train[c].astype(str).map(mapping).fillna(-1).astype(np.int32)
        X_test[c]  = X_test[c].astype(str).map(mapping).fillna(-1).astype(np.int32)

    X_test = X_test.reindex(columns=X_train.columns, fill_value=-1)

    group_cols = ["card1", "card2", "card3", "addr1", "P_emaildomain", "R_emaildomain"]
    for col in group_cols:
        if col not in X_train.columns:
            continue
        fraud_mean = y_train.groupby(X_train[col]).mean()
        X_train[f"{col}_fraud_rate"] = X_train[col].map(fraud_mean).fillna(y_train.mean())
        X_test[f"{col}_fraud_rate"]  = X_test[col].map(fraud_mean).fillna(y_train.mean())

        amt_mean = X_train.groupby(col)["TransactionAmt"].mean()
        X_train[f"{col}_amt_mean"] = X_train[col].map(amt_mean).fillna(X_train["TransactionAmt"].mean())
        X_test[f"{col}_amt_mean"]  = X_test[col].map(amt_mean).fillna(X_train["TransactionAmt"].mean())

        freq = X_train[col].value_counts()
        X_train[f"{col}_freq"] = X_train[col].map(freq).fillna(0)
        X_test[f"{col}_freq"]  = X_test[col].map(freq).fillna(0)

    if "card1_amt_mean" in X_train.columns:
        X_train["amt_vs_card1_mean"] = X_train["TransactionAmt"] - X_train["card1_amt_mean"]
        X_test["amt_vs_card1_mean"]  = X_test["TransactionAmt"]  - X_test["card1_amt_mean"]

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test  = X_test.replace([np.inf, -np.inf],  np.nan).fillna(0)

    nu = X_train.nunique(dropna=False)
    const_cols = nu[nu <= 1].index.tolist()
    X_train = X_train.drop(columns=const_cols)
    X_test  = X_test.drop(columns=[c for c in const_cols if c in X_test.columns])

    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    idx  = np.random.RandomState(42).choice(len(X_train), size=min(120_000, len(X_train)), replace=False)
    corr  = X_train.iloc[idx][num_cols].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    drop_corr = [c for c in upper.columns if (upper[c] > CORR_THRESHOLD).any()]
    protected = [c for c in drop_corr if any(kw in c for kw in ["_fraud_rate", "_amt_mean", "_freq", "amt_vs_"])]
    drop_corr = [c for c in drop_corr if c not in protected]
    X_train = X_train.drop(columns=drop_corr)
    X_test  = X_test.drop(columns=[c for c in drop_corr if c in X_test.columns])

    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    return X_test


train_txn = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
train_id  = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
train_id.columns = [c.replace('-', '_') for c in train_id.columns]
train = train_txn.merge(train_id, on="TransactionID", how="left")
X_train_raw = train.drop(columns=["isFraud", "TransactionID"])
y_train_raw = train["isFraud"].copy()
del train, train_txn, train_id
gc.collect()

X_test_processed = preprocess(X_train_raw, X_test_raw, y_train_raw)
del X_train_raw, y_train_raw
gc.collect()

print(f"✅ Test processed: {X_test_processed.shape}")
print(f"   NaNs: {X_test_processed.isnull().sum().sum()}")

✅ Test processed: (506691, 321)
   NaNs: 0


In [7]:
model = mlflow.xgboost.load_model(f"models:/{best['registry_name']}/latest")
print(f"✅ Model loaded: {type(model).__name__}")
predictions = model.predict_proba(X_test_processed)[:, 1]

print(f"Predictions shape: {predictions.shape}")
print(f"Prediction range:  [{predictions.min():.4f}, {predictions.max():.4f}]")
print(f"Mean fraud probability: {predictions.mean():.4f}")

✅ Model loaded: XGBClassifier
Predictions shape: (506691,)
Prediction range:  [0.0000, 1.0000]
Mean fraud probability: 0.1026


In [8]:
submission = pd.DataFrame({
    "TransactionID": transaction_ids,
    "isFraud":       predictions
})

submission.to_csv("/kaggle/working/submission.csv", index=False)
print(f"✅ Submission saved: {submission.shape}")
print(submission.head(10).to_string())

✅ Submission saved: (506691, 2)
   TransactionID       isFraud
0        3663549  6.961293e-05
1        3663550  6.633831e-03
2        3663551  4.525268e-07
3        3663552  2.878397e-03
4        3663553  1.702606e-03
5        3663554  5.282242e-02
6        3663555  6.352898e-02
7        3663556  4.606945e-02
8        3663557  3.765271e-05
9        3663558  7.355221e-02


In [9]:
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

assert len(submission) == len(sample_sub)
assert list(submission.columns) == list(sample_sub.columns)
assert submission["isFraud"].between(0, 1).all()

print("=== Inference Summary ===")
print(f"Best model    : {best['model']}")
print(f"Registry name : {best['registry_name']}/latest")
print(f"Val AUC       : {best['auc']:.4f}")
print(f"Model type    : {type(model).__name__}")
print(f"Features used : {X_test_processed.shape[1]}")
print(f"Predictions   : {len(submission)} transactions")
print(f"Fraud range   : {predictions.min():.4f} — {predictions.max():.4f}")
print(f"Mean fraud    : {predictions.mean():.4f}")
print("✅ Submission valid — ready to upload to Kaggle!")

=== Inference Summary ===
Best model    : XGBoost
Registry name : XGBoost_FraudDetection/latest
Val AUC       : 0.9791
Model type    : XGBClassifier
Features used : 321
Predictions   : 506691 transactions
Fraud range   : 0.0000 — 1.0000
Mean fraud    : 0.1026
✅ Submission valid — ready to upload to Kaggle!
